In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score

# Sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


In [3]:
# Carregar o dataset DoS
dos_path = Path("../data/raw/MQTT Under Attack Dataset/DoS.csv")
df = pd.read_csv(dos_path)

print(f"📊 Shape do dataset: {df.shape}")
print(f"📝 Colunas: {df.shape[1]}")
print(f"📋 Registros: {df.shape[0]}")
print(f"\n🏷️ Distribuição do Label ('type'):")
print(df['type'].value_counts())

📊 Shape do dataset: (94625, 67)
📝 Colunas: 67
📋 Registros: 94625

🏷️ Distribuição do Label ('type'):
type
normal    49111
DoS       45514
Name: count, dtype: int64


In [3]:
# Criar coluna publish_gap com NaN para todos
df['publish_gap'] = np.nan

# Calcular o intervalo apenas para linhas PUBLISH (msgtype == 3)
publish_mask = df['mqtt.msgtype'] == 3
df.loc[publish_mask, 'publish_gap'] = df.loc[publish_mask, 'frame.time_epoch'].diff()

#Utilizando foward fill para propagar último valor conhecido
df['publish_gap'] = df['publish_gap'].ffill().fillna(0)


# Verificar resultado
print(f"Registros PUBLISH: {publish_mask.sum()}")
print(f"Valores não-nulos em publish_gap: {df['publish_gap'].notna().sum()}")
print(f"\nEstatísticas do publish_gap:")
print(df['publish_gap'].describe())

Registros PUBLISH: 37863
Valores não-nulos em publish_gap: 94625

Estatísticas do publish_gap:
count    94625.000000
mean         4.097958
std         10.853330
min          0.000000
25%          0.000000
50%          0.194238
75%          2.425834
max         57.168259
Name: publish_gap, dtype: float64


In [4]:
# Criar coluna connect_gap com NaN para todos
df['connect_gap'] = np.nan

# Calcular o intervalo apenas para linhas CONNECT (msgtype == 1)
connect_mask = df['mqtt.msgtype'] == 1
df.loc[connect_mask, 'connect_gap'] = df.loc[connect_mask, 'frame.time_epoch'].diff()

# Utilizando forward fill para propagar último valor conhecido
df['connect_gap'] = df['connect_gap'].ffill().fillna(0)

# Verificar resultado
print(f"Registros CONNECT: {connect_mask.sum()}")
print(f"Valores não-nulos em connect_gap: {df['connect_gap'].notna().sum()}")
print(f"\nEstatísticas do connect_gap:")
print(df['connect_gap'].describe())

Registros CONNECT: 323
Valores não-nulos em connect_gap: 94625

Estatísticas do connect_gap:
count    94625.000000
mean         1.374774
std          3.045185
min          0.000000
25%          0.000020
50%          0.000020
75%          1.986573
max        264.861559
Name: connect_gap, dtype: float64


In [6]:


# ========== CARREGAR DADOS ==========
print("📍 Carregando dataset...")
dos_path = Path("../data/raw/MQTT Under Attack Dataset/DoS.csv")
df_full = pd.read_csv(dos_path)

print(f"📊 Shape do dataset: {df_full.shape}")
print(f"📝 Colunas: {df_full.shape[1]}")
print(f"🏷️ Distribuição do Label:")
print(df_full['type'].value_counts())

# ========== REMOVER FEATURES INDESEJADAS ==========
print("\n📍 Removendo features indesejadas...")
colunas_remover = ['mqtt.proto_len', 'mqtt.ver']
colunas_existentes = [c for c in colunas_remover if c in df_full.columns]
df_full = df_full.drop(columns=colunas_existentes, errors='ignore')

print(f"❌ Colunas removidas: {colunas_existentes}")
print(f"📊 Shape após remoção: {df_full.shape}\n")

# ========== SPLIT TREINO/TESTE ==========
print("📍 Fazendo split treino/teste...")
indices = np.arange(len(df_full))
train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.3, 
    random_state=42, 
    stratify=df_full['type']
)

df_train = df_full.iloc[train_idx].reset_index(drop=True)
df_test = df_full.iloc[test_idx].reset_index(drop=True)

print(f"✅ Treino: {len(df_train)} registros")
print(f"✅ Teste: {len(df_test)} registros")
print(f"   Proporção: {len(df_train)/(len(df_train)+len(df_test))*100:.1f}% treino / {len(df_test)/(len(df_train)+len(df_test))*100:.1f}% teste\n")

# ========== CALCULAR GAPS NO TREINO ==========
print("📍 Calculando gaps no TREINO...")

# PUBLISH GAP
df_train['publish_gap'] = np.nan
publish_mask_train = df_train['mqtt.msgtype'] == 3
df_train.loc[publish_mask_train, 'publish_gap'] = \
    df_train.loc[publish_mask_train, 'frame.time_epoch'].diff()
df_train.loc[publish_mask_train, 'publish_gap'] = \
    df_train.loc[publish_mask_train, 'publish_gap'].fillna(0)

print(f"✅ PUBLISH gap calculado (treino)")
print(f"   Registros PUBLISH: {publish_mask_train.sum()}")
print(f"   Valores não-nulos: {df_train['publish_gap'].notna().sum()}")
print(f"   Estatísticas:\n{df_train['publish_gap'].describe()}\n")

# CONNECT GAP
df_train['connect_gap'] = np.nan
connect_mask_train = df_train['mqtt.msgtype'] == 1
df_train.loc[connect_mask_train, 'connect_gap'] = \
    df_train.loc[connect_mask_train, 'frame.time_epoch'].diff()
df_train.loc[connect_mask_train, 'connect_gap'] = \
    df_train.loc[connect_mask_train, 'connect_gap'].fillna(0)

print(f"✅ CONNECT gap calculado (treino)")
print(f"   Registros CONNECT: {connect_mask_train.sum()}")
print(f"   Valores não-nulos: {df_train['connect_gap'].notna().sum()}")
print(f"   Estatísticas:\n{df_train['connect_gap'].describe()}\n")

# ========== CALCULAR GAPS NO TESTE ==========
print("📍 Calculando gaps no TESTE...")

# PUBLISH GAP
df_test['publish_gap'] = np.nan
publish_mask_test = df_test['mqtt.msgtype'] == 3
df_test.loc[publish_mask_test, 'publish_gap'] = \
    df_test.loc[publish_mask_test, 'frame.time_epoch'].diff()
df_test.loc[publish_mask_test, 'publish_gap'] = \
    df_test.loc[publish_mask_test, 'publish_gap'].fillna(0)

print(f"✅ PUBLISH gap calculado (teste)")
print(f"   Registros PUBLISH: {publish_mask_test.sum()}")
print(f"   Valores não-nulos: {df_test['publish_gap'].notna().sum()}")
print(f"   Estatísticas:\n{df_test['publish_gap'].describe()}\n")

# CONNECT GAP
df_test['connect_gap'] = np.nan
connect_mask_test = df_test['mqtt.msgtype'] == 1
df_test.loc[connect_mask_test, 'connect_gap'] = \
    df_test.loc[connect_mask_test, 'frame.time_epoch'].diff()
df_test.loc[connect_mask_test, 'connect_gap'] = \
    df_test.loc[connect_mask_test, 'connect_gap'].fillna(0)

print(f"✅ CONNECT gap calculado (teste)")
print(f"   Registros CONNECT: {connect_mask_test.sum()}")
print(f"   Valores não-nulos: {df_test['connect_gap'].notna().sum()}")
print(f"   Estatísticas:\n{df_test['connect_gap'].describe()}\n")

# ========== CONCATENAR E SALVAR ==========
print("📍 Finalizando...")
df_result = pd.concat([df_train, df_test], ignore_index=True)

print(f"✅ Gaps calculados separadamente por split")
print(f"   Total publish_gap válidos: {df_result['publish_gap'].notna().sum()}")
print(f"   Total connect_gap válidos: {df_result['connect_gap'].notna().sum()}\n")

# Salvar dataset
output_path = Path("../data/processed/DoS_with_gap_2.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_result.to_csv(output_path, index=False)

print(f"✅ Dataset salvo em: {output_path}")
print(f"📊 Shape final: {df_result.shape}")
print(f"\n🆕 Features adicionadas: 'publish_gap', 'connect_gap'")
print(f"❌ Features removidas: {colunas_existentes}")
print(f"🛡️  Método: Split TREINO/TESTE ANTES de calcular gaps (SEM DATA LEAKING)")

📍 Carregando dataset...
📊 Shape do dataset: (94625, 67)
📝 Colunas: 67
🏷️ Distribuição do Label:
type
normal    49111
DoS       45514
Name: count, dtype: int64

📍 Removendo features indesejadas...
❌ Colunas removidas: ['mqtt.proto_len', 'mqtt.ver']
📊 Shape após remoção: (94625, 65)

📍 Fazendo split treino/teste...
✅ Treino: 66237 registros
✅ Teste: 28388 registros
   Proporção: 70.0% treino / 30.0% teste

📍 Calculando gaps no TREINO...
✅ PUBLISH gap calculado (treino)
   Registros PUBLISH: 26534
   Valores não-nulos: 26534
   Estatísticas:
count    26534.000000
mean        -0.010015
std        301.214217
min       -707.485934
25%       -258.386976
50%         -0.000015
75%        258.318747
max        728.722162
Name: publish_gap, dtype: float64

✅ CONNECT gap calculado (treino)
   Registros CONNECT: 213
   Valores não-nulos: 213
   Estatísticas:
count    213.000000
mean       0.000055
std      199.833380
min     -569.721189
25%       -0.250692
50%        0.000051
75%        0.277347
ma

In [5]:
# Remover features indesejadas
colunas_remover = ['mqtt.proto_len', 'mqtt.ver']
df = df.drop(columns=[c for c in colunas_remover if c in df.columns], errors='ignore')

print(f"❌ Colunas removidas: {colunas_remover}")

# Salvar o dataset com as novas features
output_path = Path("../data/processed/DoS_with_gap.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"\n✅ Dataset salvo em: {output_path}")
print(f"📊 Shape final: {df.shape}")
print(f"\n🆕 Features adicionadas: 'publish_gap', 'connect_gap'")
print(f"❌ Features removidas: 'mqtt.proto_len', 'mqtt.ver'")

❌ Colunas removidas: ['mqtt.proto_len', 'mqtt.ver']

✅ Dataset salvo em: ../data/processed/DoS_with_gap.csv
📊 Shape final: (94625, 65)

🆕 Features adicionadas: 'publish_gap', 'connect_gap'
❌ Features removidas: 'mqtt.proto_len', 'mqtt.ver'
